<a href="https://colab.research.google.com/github/ecs-vlc/AICE3002_6001/blob/main/exercises/exercise4/seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First we import libraries and set the random seed:

In [ ]:
from torchvision.datasets.utils import download_url
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from torch import nn
from torch.nn import functional as F
from torch import optim
from tqdm.autonotebook import tqdm
from textwrap import wrap
import random
import sys
import io
import random
import numpy as np
import torch

torch.manual_seed(0)

The following block defines a dataset object which parses our data file and performs numericalisation of the sequence data:

In [ ]:
class CodeDataset(Dataset):
    def __init__(self):
        download_url('https://raw.githubusercontent.com/ecs-vlc/AICE3002_6001/refs/heads/main/exercises/exercise4/dataset.txt', '.', 'dataset.txt', None)
        with io.open('dataset.txt', 'r') as f:
            self.data = f.readlines()

        self.PAD='_'
        self.SOS='^'
        self.EOS='$'
        self.PAD_IDX=0

        # construct the vocabularies to numericalise the data
        self.alphabet = "*".join(self.PAD+self.SOS+self.EOS+"abcdefghijklmnopqrstuvwxyz ").split('*')

        self.alphabet_indices = dict((c, i) for i, c in enumerate(self.alphabet))
        self.indices_alphabet = dict((i, c) for i, c in enumerate(self.alphabet))
        self.SOS_IDX = self.alphabet_indices[self.SOS]
        self.EOS_IDX = self.alphabet_indices[self.EOS]

        self.morsebet = self.PAD+self.SOS+self.EOS+'.- /'
        self.morse_indices = dict((c, i) for i, c in enumerate(self.morsebet))
        self.indices_morse = dict((i, c) for i, c in enumerate(self.morsebet))

    def encode_alpha(self, inp):
        x = torch.zeros(len(inp), dtype=torch.long)
        for t, char in enumerate(inp):
            x[t] = self.alphabet_indices[char]

        return x

    def decode_alpha(self, ten, skip_tok=False):
        s = ''
        ten = ten.view(-1)
        for v in ten.view(-1):
            if not skip_tok:
                s += self.indices_alphabet[v.item()]
            elif v>2:
                s += self.indices_alphabet[v.item()]
        return s

    def encode_morse(self, inp):
        x = torch.zeros(len(inp), dtype=torch.long)
        for t, char in enumerate(inp):
            x[t] = self.morse_indices[char]

        return x

    def decode_morse(self, ten):
        s = ''
        for v in ten:
          s += self.indices_morse[v]
        return s

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        inp, out = self.data[i].strip().split('|')
        x = self.encode_morse(inp)
        # Reverse the target content, but retain SOS and EOS in their usual positions.
        reversed_out = self.SOS + out[1:-1][::-1] + self.EOS
        y = self.encode_alpha(reversed_out)
        return x, y


# This will be used to automatically pad all batch items to the same length
def pad_collate(batch): 
    data = [item[0] for item in batch]
    data = pad_sequence(data)
    targets = [item[1] for item in batch]
    targets = pad_sequence(targets)
    return [data, targets]

# Load the data and split randomly into training and val subsets
ds = CodeDataset()
tr, va = random_split(ds, [len(ds) - len(ds)//3, len(ds)//3])
trainloader = DataLoader(tr, batch_size=1024, shuffle=True, collate_fn=pad_collate)
valloader = DataLoader(va, batch_size=1024, shuffle=False, collate_fn=pad_collate)

We next define the model architecture, which is made up of an encoder, a decoder and the Seq2Seq class which ties everything together:

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        
        self.hid_dim = hid_dim
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim)
        
    def forward(self, src):
        # TODO
        raise NotImplementedError()


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        
        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        
    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.embedding(input)
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))        
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        
        self.encoder = encoder
        self.decoder = decoder
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(src.device)
        hidden, cell = self.encoder(src)

        # The first target token is SOS. Generation without a target is handled
        # separately by greedy_decode below.
        input = trg[0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1) 
            input = trg[t] if teacher_force else top1
        
        return outputs


INPUT_DIM = len(ds.morsebet)
OUTPUT_DIM = len(ds.alphabet)
ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
HID_DIM = 256

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM)
model = Seq2Seq(enc, dec).cuda()

Finally, we can train and evaluate the model:

In [ ]:
crit = nn.CrossEntropyLoss(ignore_index=ds.PAD_IDX)
opt = optim.Adam(model.parameters())

for e in range(10):
    model.train()

    with tqdm(total=len(trainloader), desc='train') as t:
        epoch_loss = 0
        for i, (x, y) in enumerate(trainloader):
            x = x.cuda()
            y = y.cuda()

            opt.zero_grad()
            pred = model(x, y)

            pred_dim = pred.shape[-1]
            pred = pred[1:].view(-1, pred_dim)
            y = y[1:].view(-1)

            loss = crit(pred, y)
            loss.backward()
            opt.step()

            epoch_loss = (epoch_loss*i + loss.item()) / (i+1)

            t.set_postfix(loss='{:05.3f}'.format(epoch_loss))
            t.update()

    model.eval()
    with tqdm(total=len(valloader), desc='val') as t:
        with torch.no_grad():
            epoch_loss = 0
            for i, (x, y) in enumerate(valloader):
                x = x.cuda()
                y = y.cuda()

                pred = model(x, y, teacher_forcing_ratio=0)

                pred_dim = pred.shape[-1]
                pred = pred[1:].view(-1, pred_dim)
                y = y[1:].view(-1)

                loss = crit(pred, y)
                epoch_loss = (epoch_loss*i + loss.item()) / (i+1)

                t.set_postfix(loss='{:05.3f}'.format(epoch_loss))
                t.update()

## Autoregressive inference

During training, the target supplies the number of decoding steps and may supply previous tokens through teacher forcing. At inference time there is no target sequence. The helpers below encode a Morse input, generate one token at a time, and stop once every sequence in the batch has produced EOS.

Choose `max_steps` independently of the target: using the known target length would give the model information that is unavailable during genuine inference. Predictions are padded after EOS so that a batch can be returned as one tensor. The model generates the textual content in reverse order, so `decode_predictions` reverses it for display.

The evaluation measures and length-generalisation experiments are left for you to implement.

In [ ]:
def prepare_morse(text, dataset, device):
    """Numericalise one Morse string and add SOS/EOS when necessary."""
    text = text.strip()
    if not text.startswith(dataset.SOS):
        text = dataset.SOS + ' ' + text
    if not text.endswith(dataset.EOS):
        text = text + ' ' + dataset.EOS
    return dataset.encode_morse(text).unsqueeze(1).to(device)


@torch.inference_mode()
def greedy_decode(model, src, sos_idx, eos_idx, pad_idx, max_steps=100):
    """Greedily decode a time-major batch without consulting target lengths."""
    if max_steps < 1:
        raise ValueError('max_steps must be positive')

    was_training = model.training
    model.eval()

    batch_size = src.shape[1]
    hidden, cell = model.encoder(src)
    token = torch.full((batch_size,), sos_idx, dtype=torch.long, device=src.device)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=src.device)
    predictions = []

    for _ in range(max_steps):
        logits, hidden, cell = model.decoder(token, hidden, cell)
        next_token = logits.argmax(dim=1)

        # Keep the first EOS, then pad subsequent positions for that sequence.
        emitted = torch.where(finished, torch.full_like(next_token, pad_idx), next_token)
        predictions.append(emitted)
        finished |= next_token.eq(eos_idx)

        if finished.all():
            break
        token = torch.where(finished, torch.full_like(next_token, pad_idx), next_token)

    if was_training:
        model.train()

    return torch.stack(predictions)


def decode_prediction(token_ids, dataset):
    """Convert one reversed prediction to ordinary readable text."""
    reversed_characters = []
    for token in token_ids.reshape(-1):
        index = token.item()
        if index == dataset.EOS_IDX:
            break
        if index not in (dataset.PAD_IDX, dataset.SOS_IDX):
            reversed_characters.append(dataset.indices_alphabet[index])
    return ''.join(reversed_characters)[::-1]


def decode_predictions(predictions, dataset):
    """Decode every sequence in a time-major prediction tensor."""
    return [
        decode_prediction(predictions[:, batch_index], dataset)
        for batch_index in range(predictions.shape[1])
    ]


def predict_morse(model, text, dataset, max_steps=100):
    """Convenience wrapper for inference on one Morse string."""
    device = next(model.parameters()).device
    src = prepare_morse(text, dataset, device)
    predictions = greedy_decode(
        model, src, dataset.SOS_IDX, dataset.EOS_IDX, dataset.PAD_IDX, max_steps
    )
    return decode_predictions(predictions, dataset)[0], predictions[:, 0]


# Example usage (replace the ellipsis with a Morse sequence):
# text, token_ids = predict_morse(model, '...', ds, max_steps=100)
# print(text)